# English Premier League 2024–25 — End-to-End Data Engineering Project

## 1. Project Overview

This project implements an end-to-end ETL (Extract, Transform, Load) pipeline for the English Premier League (EPL) 2024–25 season using data from the API-Football REST API.

The pipeline extracts team, fixture, and league standings data, transforms nested JSON responses into structured datasets using Python and Pandas, validates data quality, and loads the results into Microsoft SQL Server through staging tables and a dimensional data warehouse.

A post-load audit verifies warehouse integrity and confirms that the data is ready for downstream analytics.

## 2. Project Objectives

- Extract EPL team, fixture, and standings data through a REST API.
- Transform nested JSON responses into structured tabular datasets.
- Validate data completeness, uniqueness, referential integrity, and football statistics.
- Load data into SQL Server staging tables.
- Populate a dimensional data warehouse using SQL.
- Audit the warehouse after loading.
- Prepare reliable datasets for Power BI reporting.

## 3. Technology Stack

| Component | Technology |
|---|---|
| Programming language | Python |
| Data processing | Pandas |
| Data source | API-Football REST API |
| Database | Microsoft SQL Server |
| Database connectivity | pyodbc |
| Configuration and secrets | python-dotenv |
| Reporting | Power BI — planned |

## 4. Data Scope

| Dataset | Expected rows |
|---|---:|
| Teams | 20 |
| Fixtures | 380 |
| Final standings | 20 |

- **League:** English Premier League
- **League ID:** 39
- **Season parameter:** 2024
- **Season represented:** 2024–25

## 5. Pipeline Architecture

API-Football REST API → Raw JSON extraction → Python/Pandas transformation → Data validation → SQL Server staging (`stg`) → Dimensional warehouse (`dw`) → Warehouse audit → Power BI (planned)

## 6. Project Organization

The Python implementation is separated into modules under `src/`. The `main.py` script orchestrates the complete pipeline, while SQL scripts in `sql/` handle warehouse loading.

This notebook provides a documented walkthrough of the pipeline and its validation results.

## 7. Security

API credentials are stored in environment variables loaded from `.env`. Credentials and connection secrets must not be printed in notebook output or committed to version control.

## 8. Configuration and Data Extraction

### 8.1 Configuration

The pipeline loads its configuration from the project's `src.config` module. API credentials are managed through environment variables, while the league ID and season determine which EPL data is extracted.

### 8.2 Extraction

The extraction module calls the API-Football REST API to retrieve three datasets:

- **Teams:** Team identifiers, names, and venue details.
- **Fixtures:** Match dates, participating teams, scores, and match status.
- **Standings:** Final league positions, points, wins, draws, losses, and goal statistics.

The original API responses are saved as JSON files under `data/raw/`, preserving the extracted source data for traceability and troubleshooting.

In [1]:
##Display safe configuration details


from src import config

print("Project directory:", config.PROJECT_DIR)
print("League ID:", config.LEAGUE_ID)
print("Season parameter:", config.SEASON)
print("API key configured:", bool(config.API_KEY))
print("SQL Server configured:", bool(config.SERVER))
print("Database configured:", bool(config.DATABASE))


Project directory: C:\Users\hp\OneDrive\Documents\Data_Analytics\Python\EPL_Analytics
League ID: 39
Season parameter: 2024
API key configured: True
SQL Server configured: True
Database configured: True


In [2]:
##Extract the datasets


import logging
from src.extract import extract_all

logging.basicConfig(level=logging.INFO)

raw_datasets = extract_all()

for name, data in raw_datasets.items():
    print(f"{name.title()}: {data.get('results', 0)} API results")


INFO:src.extract:Starting EPL extraction for season 2024
INFO:src.extract:Fetched endpoint=teams, results=20
INFO:src.extract:Saved raw response to C:\Users\hp\OneDrive\Documents\Data_Analytics\Python\EPL_Analytics\data\raw\teams_2024_20261029T100433Z.json
INFO:src.extract:Fetched endpoint=fixtures, results=380
INFO:src.extract:Saved raw response to C:\Users\hp\OneDrive\Documents\Data_Analytics\Python\EPL_Analytics\data\raw\fixtures_2024_20261029T100433Z.json
INFO:src.extract:Fetched endpoint=standings, results=1
INFO:src.extract:Saved raw response to C:\Users\hp\OneDrive\Documents\Data_Analytics\Python\EPL_Analytics\data\raw\standings_2024_20261029T100434Z.json
INFO:src.extract:EPL extraction completed successfully


Teams: 20 API results
Fixtures: 380 API results
Standings: 1 API results


In [3]:
##Check the extracted response structure


for name, data in raw_datasets.items():
    response = data.get("response", [])

    print(f"\n{'=' * 12} {name.upper()} {'=' * 12}")
    print("API result count:", data.get("results", 0))
    print("Response items:", len(response))

    if response:
        print("First response item keys:", list(response[0].keys()))




============ TEAMS ============
API result count: 20
Response items: 20
First response item keys: ['team', 'venue']

============ FIXTURES ============
API result count: 380
Response items: 380
First response item keys: ['fixture', 'league', 'teams', 'goals', 'score']

============ STANDINGS ============
API result count: 1
Response items: 1
First response item keys: ['league']


## 9. Data Transformation

The API returns nested JSON responses. These responses must be transformed into structured tables before they can be validated and loaded into SQL Server.

The transformation layer uses Python and Pandas to flatten the relevant fields, standardize column names, and prepare three datasets:

- **Teams:** Team identifiers, team names, and venue attributes.
- **Fixtures:** Match identifiers, dates, home and away teams, scores, and match status.
- **Standings:** Team positions, points, wins, draws, losses, goals scored, and goals conceded.

The transformation logic is maintained in `src/transform.py` so that it can be reused by both the notebook and the production pipeline.

In [4]:
##Transform the extracted data


from src.transform import transform_all

transformed = transform_all(raw_datasets)

print("Transformation completed successfully.")


INFO:numexpr.utils:NumExpr defaulting to 8 threads.


Transformation completed successfully.


In [5]:
##Inspect the transformed datasets


for name, df in transformed.items():
    print(f"\n{'=' * 15} {name.upper()} {'=' * 15}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", df.columns.tolist())
    display(df.head(3))



=============== TEAMS ===============
Rows: 20
Columns: 13
Column names: ['team_id', 'team_name', 'team_code', 'country', 'founded', 'national', 'team_logo', 'venue_id', 'venue_name', 'venue_address', 'venue_city', 'venue_capacity', 'venue_surface']


,team_id,team_name,team_code,country,founded,national,team_logo,venue_id,venue_name,venue_address,venue_city,venue_capacity,venue_surface
0,33,Manchester United,MUN,England,1878,False,https://media.api-sports.io/football/teams/33.png,556,Old Trafford,Sir Matt Busby Way,Manchester,76212,grass
1,34,Newcastle,NEW,England,1892,False,https://media.api-sports.io/football/teams/34.png,562,St. James' Park,St. James&apos; Street,Newcastle upon Tyne,52758,grass
2,35,Bournemouth,BOU,England,1899,False,https://media.api-sports.io/football/teams/35.png,504,Vitality Stadium,"Dean Court, Kings Park","Bournemouth, Dorset",12000,grass



=============== FIXTURES ===============
Rows: 380
Columns: 25
Column names: ['fixture_id', 'league_id', 'season', 'round', 'fixture_date', 'timestamp', 'referee', 'venue_id', 'venue_name', 'venue_city', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name', 'home_goals', 'away_goals', 'home_fulltime_goals', 'away_fulltime_goals', 'home_halftime_goals', 'away_halftime_goals', 'home_winner', 'away_winner', 'status_short', 'status_long', 'elapsed_minutes']


,fixture_id,league_id,season,round,fixture_date,timestamp,referee,venue_id,venue_name,venue_city,...,away_goals,home_fulltime_goals,away_fulltime_goals,home_halftime_goals,away_halftime_goals,home_winner,away_winner,status_short,status_long,elapsed_minutes
0,1208021,39,2024,Regular Season - 1,2024-08-16 19:00:00+00:00,1723834800,R. Jones,556,Old Trafford,Manchester,...,0,1,0,0,0,True,False,FT,Match Finished,90
1,1208022,39,2024,Regular Season - 1,2024-08-17 11:30:00+00:00,1723894200,T. Robinson,545,Portman Road,"Ipswich, Suffolk",...,2,0,2,0,0,False,True,FT,Match Finished,90
2,1208025,39,2024,Regular Season - 1,2024-08-17 14:00:00+00:00,1723903200,C. Pawson,562,St. James' Park,Newcastle upon Tyne,...,0,1,0,1,0,True,False,FT,Match Finished,90



=============== STANDINGS ===============
Rows: 20
Columns: 16
Column names: ['team_id', 'team_name', 'season', 'position', 'points', 'goal_difference', 'played', 'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'form', 'description', 'group_name', 'updated_at']


,team_id,team_name,season,position,points,goal_difference,played,wins,draws,losses,goals_for,goals_against,form,description,group_name,updated_at
0,40,Liverpool,2024,1,84,45,38,25,9,4,86,41,DLDLW,Champions League,Premier League,2025-05-26T00:00:00+00:00
1,42,Arsenal,2024,2,74,35,38,20,14,4,69,34,WWDLD,Champions League,Premier League,2025-05-26T00:00:00+00:00
2,50,Manchester City,2024,3,71,28,38,21,8,9,72,44,WWDWW,Champions League,Premier League,2025-05-26T00:00:00+00:00


In [6]:
##Check data types and missing values


for name, df in transformed.items():
    print(f"\n{'=' * 15} {name.upper()} {'=' * 15}")

    print("\nData types:")
    display(df.dtypes.rename("data_type").to_frame())

    print("\nMissing values by column:")
    display(
        df.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
        .query("missing_count > 0")
    )



=============== TEAMS ===============

Data types:


,data_type
team_id,int64
team_name,str
team_code,str
country,str
founded,int64
national,bool
team_logo,str
venue_id,int64
venue_name,str
venue_address,str



Missing values by column:


,missing_count



=============== FIXTURES ===============

Data types:


,data_type
fixture_id,int64
league_id,int64
season,int64
round,str
fixture_date,"datetime64[us, UTC]"
timestamp,int64
referee,str
venue_id,int64
venue_name,str
venue_city,str


... (2 outputs left out)


=============== STANDINGS ===============

Data types:


,data_type
team_id,int64
team_name,str
season,int64
position,int64
points,int64
goal_difference,int64
played,int64
wins,int64
draws,int64
losses,int64



Missing values by column:


,missing_count
description,8


## 10. Data Quality and Validation

Before loading data into SQL Server, the pipeline validates the transformed datasets to identify missing identifiers, duplicate records, invalid relationships, and inconsistent football statistics.

The validation logic is implemented in `src/validate.py` and is reused by the production pipeline.

Key checks include:

- **Completeness:** Required datasets contain records and essential identifiers are present.
- **Uniqueness:** Team IDs, fixture IDs, and team-season combinations are unique.
- **Referential integrity:** Fixture and standings records reference teams present in the team dataset.
- **Football statistics:** Points equal three times wins plus draws; goal difference equals goals scored minus goals conceded; matches played equal wins plus draws plus losses.
- **Fixture integrity:** Home and away teams differ, and fixture data contains valid dates and season information.

The pipeline stops if a required validation fails, preventing invalid data from proceeding to the loading stages.

In [7]:
##Run the project's validation module


from src.validate import validate_all

validation_passed = validate_all(transformed)

print("Validation passed:", validation_passed)


INFO:src.validate:PASS: Teams dataset is not empty
INFO:src.validate:PASS: Team IDs are not null
INFO:src.validate:PASS: Team IDs are unique
INFO:src.validate:PASS: Team names are not null
INFO:src.validate:PASS: Fixtures dataset is not empty
INFO:src.validate:PASS: Fixture IDs are not null
INFO:src.validate:PASS: Fixture IDs are unique
INFO:src.validate:PASS: Fixture dates are not null
INFO:src.validate:PASS: Home and away teams differ
INFO:src.validate:PASS: All home team IDs exist in teams
INFO:src.validate:PASS: All away team IDs exist in teams
INFO:src.validate:PASS: Fixture seasons are not null
INFO:src.validate:PASS: Standings dataset is not empty
INFO:src.validate:PASS: Standing team IDs are not null
INFO:src.validate:PASS: Standing seasons are not null
INFO:src.validate:PASS: Team-season standings keys are unique
INFO:src.validate:PASS: All standings team IDs exist in teams
INFO:src.validate:PASS: Standings points equal wins * 3 + draws
INFO:src.validate:PASS: Goal difference 

Validation passed: True


In [8]:
##Add a compact summary of key data-quality checks


teams_df = transformed["teams"]
fixtures_df = transformed["fixtures"]
standings_df = transformed["standings"]

quality_checks = {
    "Expected team count": len(teams_df) == 20,
    "Unique team IDs": teams_df["team_id"].is_unique,
    "Expected fixture count": len(fixtures_df) == 380,
    "Unique fixture IDs": fixtures_df["fixture_id"].is_unique,
    "Expected standings count": len(standings_df) == 20,
    "Unique team-season keys": not standings_df.duplicated(
        ["team_id", "season"]
    ).any(),
    "Valid points calculation": (
        standings_df["points"]
        == standings_df["wins"] * 3 + standings_df["draws"]
    ).all(),
    "Valid goal difference": (
        standings_df["goal_difference"]
        == standings_df["goals_for"] - standings_df["goals_against"]
    ).all(),
    "Valid matches played": (
        standings_df["played"]
        == standings_df["wins"] + standings_df["draws"] + standings_df["losses"]
    ).all(),
}

for check, passed in quality_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {check}")

assert all(quality_checks.values()), "One or more quality checks failed."


PASS | Expected team count
PASS | Unique team IDs
PASS | Expected fixture count
PASS | Unique fixture IDs
PASS | Expected standings count
PASS | Unique team-season keys
PASS | Valid points calculation
PASS | Valid goal difference
PASS | Valid matches played


## 11. SQL Server Data Loading

### 11.1 Staging Layer

The transformed datasets are loaded into SQL Server staging tables:

- `stg.Teams`
- `stg.Fixtures`
- `stg.Standings`

The staging layer holds the latest extracted dataset before it is processed into the warehouse. The staging loader prepares the data types, refreshes the staging tables within a transaction, verifies the loaded row counts, and commits the changes only when the load succeeds.

### 11.2 Dimensional Data Warehouse

The warehouse layer organizes data into dimension and fact tables:

- `dw.DimTeam` — team and venue attributes.
- `dw.FactFixture` — match dates, participating teams, match status, and scores.
- `dw.FactStandings` — final league positions and team performance statistics.

The warehouse-loading logic is maintained in `sql/WarehouseLoad.sql` and executed by `src/load_warehouse.py`.

The script validates staging data, updates team and fixture records, refreshes standings for the selected season, and uses database transactions to protect the integrity of the warehouse load.

In [9]:
##Load the staging tables


from src.load_staging import load_staging

staging_summary = load_staging(transformed)

print("Staging load summary:")
for table, row_count in staging_summary.items():
    print(f"{table}: {row_count} rows loaded")


INFO:src.load_staging:Inserted 20 rows into stg.Teams
INFO:src.load_staging:Inserted 380 rows into stg.Fixtures
INFO:src.load_staging:Inserted 20 rows into stg.Standings
INFO:src.load_staging:Verified stg.Teams: 20 rows
INFO:src.load_staging:Verified stg.Fixtures: 380 rows
INFO:src.load_staging:Verified stg.Standings: 20 rows
INFO:src.load_staging:Staging refresh committed successfully.


Staging load summary:
stg.Teams: 20 rows loaded
stg.Fixtures: 380 rows loaded
stg.Standings: 20 rows loaded


### 11.3 Loading the Warehouse

After staging is loaded, the warehouse script validates the staged records and applies the changes to the dimensional tables. The season parameter is supplied by the project configuration, allowing the pipeline to target the intended season without hardcoding it in the Python orchestration code.

In [10]:

from src.load_warehouse import load_warehouse

warehouse_summary = load_warehouse()

print("Warehouse load summary:")
for item in warehouse_summary:
    for key, value in item.items():
        print(f"{key}: {value}")
    print("-" * 30)


INFO:src.load_warehouse:Starting warehouse upsert for season 2024
INFO:src.load_warehouse:Warehouse upsert completed for season 2024


Warehouse load summary:
season: 2024
staged_teams: 20
staged_fixtures: 380
staged_standings: 20
warehouse_teams: 20
warehouse_fixtures_for_season: 380
warehouse_standings_for_season: 20
------------------------------


## 12. Warehouse Audit and Verification

After loading the warehouse, the pipeline performs an audit to confirm that the stored data meets key integrity requirements.

The audit checks:

- Expected counts for teams, fixtures, and standings.
- Duplicate team and fixture identifiers.
- Fixture references to missing teams.
- Consistency of points, goal difference, and matches played.
- Duplicate league positions.

The audit is implemented in `src/audit.py`. If an audit check fails, the production pipeline reports failure rather than treating the run as successful.

These checks provide an additional verification layer after the transformation and warehouse-loading stages.

In [11]:
##Run the warehouse audit


from src.audit import run_warehouse_audit

audit_results = run_warehouse_audit()

for name, value in audit_results.items():
    print(f"{name}: {value}")

assert audit_results["audit_passed"], "Warehouse audit failed."


INFO:src.audit:Warehouse audit passed.


warehouse_teams: 20
warehouse_fixtures: 380
warehouse_standings: 20
duplicate_team_ids: 0
duplicate_fixture_ids: 0
missing_fixture_teams: 0
invalid_standings: 0
duplicate_standings_positions: 0
audit_checks: {'duplicate_team_ids': True, 'duplicate_fixture_ids': True, 'missing_fixture_teams': True, 'invalid_standings': True, 'duplicate_standings_positions': True}
audit_passed: True


## 13. Project Results and Conclusion

The EPL 2024–25 ETL pipeline has been implemented and successfully executed using Python and Microsoft SQL Server.

### Verified Results

| Metric | Result |
|---|---:|
| Teams extracted and loaded | 20 |
| Fixtures extracted and loaded | 380 |
| Final standings records loaded | 20 |
| Duplicate team IDs found by audit | 0 |
| Duplicate fixture IDs found by audit | 0 |
| Missing fixture team references | 0 |
| Invalid standings records found by audit | 0 |

The end-to-end pipeline completed successfully, and the warehouse audit passed for the 2024–25 season.

### Key Engineering Practices Demonstrated

- Modular Python development.
- REST API data extraction and raw JSON preservation.
- Pandas-based data transformation.
- Data quality validation before loading.
- SQL Server staging and dimensional warehouse design.
- Transaction-aware database loading.
- Post-load integrity auditing.
- Centralized pipeline orchestration and logging.

### Next Phase

The next phase is to build Power BI reports using the validated warehouse tables and analytical views, focusing on league standings, team performance, match outcomes, and home-versus-away performance.

The notebook documents the pipeline; `main.py` remains the primary script for executing the full workflow.

In [12]:

from src import config

project_root = config.PROJECT_DIR

print("PROJECT ROOT")
print(project_root)

print("\nPROJECT FILES")
for path in sorted(project_root.iterdir()):
    if path.is_file():
        print(f"FILE   {path.name}")
    elif path.is_dir() and path.name != ".git":
        print(f"FOLDER {path.name}")

print("\nSQL FILES")
sql_folder = project_root / "sql"

if sql_folder.exists():
    for path in sorted(sql_folder.iterdir()):
        if path.is_file():
            print(path.name)
else:
    print("SQL folder not found")


PROJECT ROOT
C:\Users\hp\OneDrive\Documents\Data_Analytics\Python\EPL_Analytics

PROJECT FILES
FILE   .env
FILE   .gitignore
FOLDER .ipynb_checkpoints
FOLDER data
FILE   EPL.ipynb
FILE   EPL_ETL.ipynb
FOLDER logs
FILE   main.py
FILE   readme.md
FILE   requirements.txt
FOLDER sql
FOLDER src

SQL FILES
WarehouseLoad.sql


In [13]:

import pyodbc
from src import config

with pyodbc.connect(config.CONNECTION_STRING) as conn:
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            TABLE_SCHEMA,
            TABLE_NAME,
            TABLE_TYPE
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA IN ('stg', 'dw')
        ORDER BY TABLE_SCHEMA, TABLE_NAME;
    """)

    print("DATABASE OBJECTS")
    for row in cursor.fetchall():
        print(f"{row.TABLE_SCHEMA}.{row.TABLE_NAME} | {row.TABLE_TYPE}")

    cursor.execute("""
        SELECT
            TABLE_SCHEMA,
            TABLE_NAME
        FROM INFORMATION_SCHEMA.VIEWS
        WHERE TABLE_SCHEMA = 'dw'
        ORDER BY TABLE_NAME;
    """)

    print("\nWAREHOUSE VIEWS")
    for row in cursor.fetchall():
        print(f"{row.TABLE_SCHEMA}.{row.TABLE_NAME}")


DATABASE OBJECTS
dw.DimTeam | BASE TABLE
dw.FactFixture | BASE TABLE
dw.FactStandings | BASE TABLE
dw.vw_MatchAnalysis | VIEW
dw.vw_StandingsAnalysis | VIEW
dw.vw_TeamMatchPerformance | VIEW
stg.Fixtures | BASE TABLE
stg.Standings | BASE TABLE
stg.Teams | BASE TABLE

WAREHOUSE VIEWS
dw.vw_MatchAnalysis
dw.vw_StandingsAnalysis
dw.vw_TeamMatchPerformance


In [14]:

import pyodbc
from src import config

with pyodbc.connect(config.CONNECTION_STRING) as conn:
    cursor = conn.cursor()

    print("=== TABLE COLUMN DEFINITIONS ===")

    cursor.execute("""
        SELECT
            TABLE_SCHEMA,
            TABLE_NAME,
            ORDINAL_POSITION,
            COLUMN_NAME,
            DATA_TYPE,
            IS_NULLABLE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA IN ('stg', 'dw')
        ORDER BY TABLE_SCHEMA, TABLE_NAME, ORDINAL_POSITION;
    """)

    for row in cursor.fetchall():
        print(
            f"{row.TABLE_SCHEMA}.{row.TABLE_NAME} | "
            f"{row.ORDINAL_POSITION}. {row.COLUMN_NAME} "
            f"({row.DATA_TYPE}, nullable={row.IS_NULLABLE})"
        )

    print("\n=== EXISTING VIEW DEFINITIONS ===")

    cursor.execute("""
        SELECT
            s.name AS schema_name,
            v.name AS view_name,
            OBJECT_DEFINITION(v.object_id) AS definition
        FROM sys.views AS v
        INNER JOIN sys.schemas AS s
            ON v.schema_id = s.schema_id
        WHERE s.name = 'dw'
        ORDER BY v.name;
    """)

    for row in cursor.fetchall():
        print(f"\n--- {row.schema_name}.{row.view_name} ---")
        print(row.definition or "Definition unavailable.")


=== TABLE COLUMN DEFINITIONS ===
dw.DimTeam | 1. team_id (int, nullable=NO)
dw.DimTeam | 2. team_name (nvarchar, nullable=NO)
dw.DimTeam | 3. team_code (nvarchar, nullable=YES)
dw.DimTeam | 4. team_country (nvarchar, nullable=YES)
dw.DimTeam | 5. team_founded (int, nullable=YES)
dw.DimTeam | 6. venue_id (int, nullable=YES)
dw.DimTeam | 7. venue_name (nvarchar, nullable=YES)
dw.DimTeam | 8. venue_city (nvarchar, nullable=YES)
dw.DimTeam | 9. venue_capacity (int, nullable=YES)
dw.DimTeam | 10. venue_surface (nvarchar, nullable=YES)
dw.FactFixture | 1. fixture_id (int, nullable=NO)
dw.FactFixture | 2. season (int, nullable=NO)
dw.FactFixture | 3. fixture_date (datetimeoffset, nullable=NO)
dw.FactFixture | 4. venue_name (nvarchar, nullable=YES)
dw.FactFixture | 5. venue_city (nvarchar, nullable=YES)
dw.FactFixture | 6. match_status (nvarchar, nullable=NO)
dw.FactFixture | 7. league_round (nvarchar, nullable=YES)
dw.FactFixture | 8. home_team_id (int, nullable=NO)
dw.FactFixture | 9. away_t

In [ ]:
%run "C:\Users\hp\OneDrive\Documents\Data_Analytics\Python\EPL_Analytics\main.py"

##to run the main.py file